## Instructions
Now that the e-commerse DB is ready its time to test it.

In [2]:
import sqlite3
import pandas as pd

## Basic

### Volume

In [3]:
#What is the total number of orders registered in the system?  
conn = sqlite3.connect('data_base/e-commerse.db')

query_1 = '''
SELECT 
    COUNT(DISTINCT o.id) AS total_orders
FROM 
    Orders o;
'''

#What is the total historical amount collected?
query_2 = '''
SELECT 
    SUM(o.total_amount) AS total_historic_amount_received_round
FROM 
    Orders o;
'''

response_1 = pd.read_sql_query(query_1, conn)
response_2 = pd.read_sql_query(query_2, conn)

response_2['total_historic_amount_received_round'] = response_2['total_historic_amount_received_round'].astype(int)

display(response_1)
display(response_2)

conn.close()

,total_orders
0,22000


,total_historic_amount_received_round
0,193774175


### Filter

In [4]:
# list the ID, name, price of the 15 most expensive products in store
conn = sqlite3.connect('data_base/e-commerse.db')

query = '''
SELECT 
    p.id,
    p.name,
    p.price
FROM 
    Products p
ORDER BY p.price DESC
LIMIT 15;
'''

response = pd.read_sql_query(query, conn)
display(response)

conn.close()

,id,name,price
0,171,Pacifica Touring,37306.01
1,170,Durango SXT RWD,36597.01
2,168,Charger SXT RWD,36536.77
3,167,300 Touring,29358.46
4,169,Dodge Hornet GT Plus,25178.29
5,115,MotoGP CI.H1,17411.66
6,191,Rolex Cellini Moonphase,16845.01
7,98,Rolex Submariner Watch,15069.66
8,96,Rolex Cellini Moonphase,13571.42
9,97,Rolex Datejust,12970.60


### First cross

In [5]:
# Obtain the name of the users and the dates in witch they did their first order
conn = sqlite3.connect('data_base/e-commerse.db')

query = '''
SELECT
    u.id,
    u.name,
    MIN(o.order_date) AS first_order_at
FROM 
    Users u
JOIN Orders o ON u.id = o.user_id
GROUP BY u.id, u.name;
'''

response = pd.read_sql_query(query, conn)
display(response)

conn.close()


,id,name,first_order_at
0,2,Sofia Perez,2024-11-11 00:00:00
1,3,Amelia Sanchez,2025-09-06 00:00:00
2,4,Noah Diaz,2024-03-31 00:00:00
3,5,Catalina Brown,2024-11-16 00:00:00
4,6,Mateo Romero,2024-03-19 00:00:00
...,...,...,...
3066,3496,Camila Martinez,2025-08-21 00:00:00
3067,3497,Liam Benitez,2025-09-25 00:00:00
3068,3498,Valentina Fernandez,2024-08-25 00:00:00
3069,3499,Camila Mendez,2024-07-23 00:00:00


## Intermediate Level (Groupings and Joins)

### Average Ticket: What is the average spend (total_amount) per order for purchases made in 2024?

In [6]:
conn = sqlite3.connect('data_base/e-commerse.db')

query = '''
SELECT 
    AVG(o.total_amount) AS avegare_spend_in_2024
FROM 
    Orders o
WHERE o.order_date BETWEEN '2024-01-01' AND '2024-12-31';
'''

response = pd.read_sql_query(query, conn)
display(response)

conn.close()

,avegare_spend_in_2024
0,9096.249336


### Best Sellers: List the ID and name of the 5 products that have sold the most units in history (adding the quantities in order_items).

In [7]:
conn = sqlite3.connect('data_base/e-commerse.db')

query = '''
SELECT
    p.id,
    p.name,
    SUM(oi.quantity) as total_sold
FROM 
    Products p 
JOIN OrderItems oi ON p.id = oi.product_id
GROUP BY p.id, p.name 
ORDER BY total_sold DESC
LIMIT 5;
'''

response = pd.read_sql_query(query, conn)
display(response)

conn.close()

,id,name,total_sold
0,155,Classic Sun Glasses,3499
1,42,Water,3187
2,151,Tennis Ball,2936
3,53,Chopping Board,2798
4,157,Party Glasses,2351


### Usuarios Inactivos: Encontrar el nombre y el email de todos los usuarios registrados que nunca han realizado una compra.

In [13]:
conn = sqlite3.connect('data_base/e-commerse.db')

query = '''
SELECT
    u.id,
    u.name,
    u.email
FROM Users u
WHERE u.id NOT IN (SELECT DISTINCT o.user_id FROM Orders o);
'''

response = pd.read_sql_query(query, conn)
display(response)

conn.close()

,id,name,email
0,1,Leo Pereira,leo.pereira@hotmail.com
1,12,Benjamin Gonzalez,benjamin.gonzalez@outlook.com
2,27,Julia Lopez,julia.lopez@zolver.com
3,32,Emma Alvarez,emma.alvarez@yahoo.com
4,39,Noah Pereira,noah.pereira@yahoo.com
...,...,...,...
424,3434,Oliver Garcia,oliver.garcia4@yahoo.com
425,3441,Elena Garcia,elena.garcia5@yahoo.com
426,3451,Catalina Diaz,catalina.diaz7@hotmail.com
427,3457,Olivia Ruiz,olivia.ruiz7@mail.com


## Advanced Level (Total Integration)

### Line Analysis: Write a query that returns the order ID (order_id), the total number of distinct items in that order, and validates if the mathematical sum of (quantity * unit_price) in its lines matches the total_amount in the header.

In [26]:
conn = sqlite3.connect('data_base/e-commerse.db')

query = '''
SELECT 
    oi.order_id,
    SUM(oi.quantity * oi.unit_price) AS total_amount_calc,
    o.total_amount AS total_amount_tab,
    CASE 
        WHEN SUM(oi.quantity * oi.unit_price) = o.total_amount THEN 'correct price'
        ELSE 'incorrect price'
    END AS price_status
FROM OrderItems oi
JOIN Orders o ON oi.order_id = o.id
GROUP BY oi.order_id;
'''

response = pd.read_sql_query(query, conn)

display(response)

conn.close()



,order_id,total_amount_calc,total_amount_tab,price_status
0,1,46.16,46.16,correct price
1,2,18091.80,18091.80,correct price
2,3,57.48,57.48,incorrect price
3,4,352.88,352.88,correct price
4,5,58.07,58.07,correct price
...,...,...,...,...
21995,21996,5.35,5.35,correct price
21996,21997,31.94,31.94,correct price
21997,21998,931.05,931.05,incorrect price
21998,21999,14.90,14.90,correct price


### Notes
Some orders price_status shows as incorrect price, but for what i can see they are not, i need to see more into that